In [ ]:
# Fabric notebook source
# ============================================================================
# Metadata-driven view deployment for Microsoft Fabric  --  TEMPLATE
# ----------------------------------------------------------------------------
# Runtime : Fabric PySpark notebook, run IN THE TARGET workspace
# Reads   : three metadata tables produced by the companion extract notebook
# Writes  : views into the lakehouses/warehouses named in that metadata,
#           plus a deployment log table
#
# No lakehouse, schema, table or view name is hardcoded anywhere below. The
# set of target items is derived from the metadata and validated against the
# items that actually exist in the target workspace. The only names you set
# are the metadata table locations in Cell 1.
#
# Spark reads the metadata. DDL executes over TDS via pyodbc, because Spark
# SQL CREATE VIEW creates Spark metastore views, not SQL analytics endpoint
# views. There is no way around this split for lakehouse views.
#
# ---------------------------------------------------------------------------
# THE METADATA CONTRACT THIS VERSION READS
# ---------------------------------------------------------------------------
#   view_definitions : node_id, item_name, schema_name, view_name,
#                      view_definition, extracted_at_utc
#   dependencies     : source_id, target_id, source_node_id, resolved_type,
#                      extracted_at_utc
#   objects          : item_name, schema_name, object_type, item_type,
#                      extracted_at_utc
#
# source_node_id, resolved_type and item_type are treated as optional -- an
# older extract without them degrades to the previous behaviour and says so.
# ---------------------------------------------------------------------------
#
# Twelve behaviours here exist because of real failures. Do not simplify them:
#
#  1. CONNECTIONS ARE THREAD-LOCAL. Fabric does not support MARS (Multiple
#     Active Result Sets), so one connection cannot carry concurrent
#     statements. Sharing a connection across a thread pool fails with
#     "Connection is busy with results for another command".
#
#  2. EDGES RESOLVE THROUGH source_node_id, NOT source_id. The extract
#     suffixes node_id with #VIEW / #USER_TABLE when a name collides
#     case-insensitively within an item. target_id carries that suffix;
#     source_id does not. Matching source_id against node_id therefore drops
#     every edge whose source is a suffixed view, flattening the graph and
#     scheduling dependent views in the same wave as their sources.
#     referenced_type is still deliberately not consulted: cross-database
#     references have a NULL referenced_id and arrive as UNRESOLVED even when
#     the target exists.
#
#  3. GRAPH KEYS ARE LOWERCASED. sys.sql_expression_dependencies records
#     referenced names as WRITTEN in the referencing view's SQL, not as the
#     object is named. Original casing is preserved for execution.
#
#  4. IDENTIFIER CASING IS NORMALIZED AGAINST THE LIVE CATALOG. Fabric SQL
#     endpoints default to a case-sensitive collation, so DDL that worked in a
#     case-insensitive source environment fails here. Rewrites are validated
#     against sys.objects/sys.schemas, never guessed.
#
#  5. SCHEMAS ARE NOT CREATED BY DEFAULT, and warehouse schemas are never
#     treated as pipeline-owned. In a Lakehouse a schema holding tables must
#     come from the Spark side; in a Warehouse every schema is a T-SQL schema
#     whether it holds tables or not, so the table-bearing rule must not apply
#     to it -- otherwise the run demands pipelines that do not exist.
#
#  6. EACH METADATA TABLE PICKS ITS OWN LATEST SNAPSHOT, AND A FILTER THAT
#     EMPTIES A NON-EMPTY TABLE BLOCKS THE RUN. Taking max(extracted_at_utc)
#     from one table and applying that literal to the others fails silently
#     when the writers disagree on precision: a loader that round-trips
#     through Excel or pandas stores 10:51:59.305 where Spark stored
#     10:51:59.305652, and .305 == .305652 is false. The dependency set then
#     reads as empty, the graph flattens to one level, and every view that
#     reads another view in the same run fails at CREATE.
#
#  7. DUPLICATE node_id BLOCKS THE RUN. The lookup dict would otherwise keep
#     whichever row landed last, deploying one definition and discarding the
#     other with no signal.
#
#  8. AN EMPTY DEPENDENCY GRAPH BLOCKS THE RUN, and the stored definitions are
#     parsed as a second source of edges. sys.sql_expression_dependencies
#     routinely under-reports: 126 rows can yield 24 usable view-to-view
#     edges, stranding views that read each other in the same wave.
#
#  9. VIEWS THIS RUN CREATES ARE SEEDED INTO THE CASING CATALOG, BOUND TO THE
#     LIVE SCHEMA SPELLING. Without a seed, Cell 5b can never correct a
#     reference to a view created later in the same run. But the seed must use
#     the schema name the TARGET uses, not the one the metadata carries: two
#     lakehouses in one estate can disagree on schema casing (LakehouseA.contoso
#     vs LakehouseB.Contoso), and seeding the metadata's spelling carries
#     the wrong one across that boundary.
#
# 10. THE CREATE CLAUSE IS REWRITTEN ONCE, BY fix_create ALONE. Running the
#     reference pass over the whole definition lets it re-resolve the CREATE
#     target that fix_create just corrected and revert it, producing error
#     2760 against a schema that does not exist. The reference pass therefore
#     sees only the body.
#
# 11. A VIEW WITH A KNOWN-MISSING SOURCE IS SKIPPED, NOT ATTEMPTED. Attempting
#     it drops the existing view and then fails to recreate it, so a
#     diagnosable metadata gap turns into lost state in the target.
#
# 12. COLUMNS ARE NEVER REWRITTEN. A column identifier appears in select
#     lists, predicates, aliases, CTE scopes and function names with no
#     reliable way to tell them apart by regex. A rewriter that merges the
#     column vocabularies of every referenced object will recase a CTE-scoped
#     column to a base-table spelling and break SQL that was correct. Column
#     mismatches are fixed at ingestion -- see the note above Cell 6.
#
# WARNING: each view is DROPped then CREATEd. If CREATE fails the view is left
# dropped, so a failed run can leave the target missing views it had before.
# Re-running after fixing the root cause is the repair.
#
# NOT_VERIFIED as a template: exercise it with DRY_RUN = True in a non-
# production workspace first.
# ============================================================================

# CELL ********************
# ## Cell 1 - configuration (the only place you set anything)

from datetime import datetime, timezone

# Where the extract notebook wrote its output. Adjust to your own layout.
METADATA_SCHEMA = "metadata"
VIEWS_TABLE = f"{METADATA_SCHEMA}.view_definitions"
OBJECTS_TABLE = f"{METADATA_SCHEMA}.objects"
DEPENDENCIES_TABLE = f"{METADATA_SCHEMA}.dependencies"
LOG_TABLE = f"{METADATA_SCHEMA}.deployment_log"

# --- snapshot selection -----------------------------------------------------
# Deploy only an approved snapshot. Accepts a datetime or an ISO string.
# Under SNAPSHOT_MATCH_MODE = "exact" the literal must match to the
# microsecond; "millisecond" tolerates a lossy loader.
APPROVED_SNAPSHOT = None              # e.g. "2026-08-08 10:51:59.305"

# When APPROVED_SNAPSHOT is None: True reads the newest extract only, False
# reads every snapshot at once. False is almost never what you want on
# append-mode metadata tables -- it duplicates the plan.
LATEST_SNAPSHOT_ONLY = True

SNAPSHOT_MATCH_MODE = "exact"         # "exact" | "millisecond"
SNAPSHOT_TOLERANCE_SEC = 1800         # max legitimate spread between tables

# --- scope ------------------------------------------------------------------
INCLUDE_ITEMS = None                  # e.g. {"MyGoldLakehouse"}
EXCLUDE_ITEMS = None                  # sandboxes, scratch lakehouses
EXCLUDE_VIEWS = None                  # node_id values, case-insensitive

TARGET_WORKSPACE = None               # None => this notebook's workspace
DRY_RUN = False                        # True => validate and plan only

# --- graph ------------------------------------------------------------------
REQUIRE_DEPENDENCY_EDGES = True       # block on a flat graph. See note 8.

# "off" | "fallback" | "always". "always" unions DDL-parsed edges with the
# dependency table, which is the safe default given how often the table
# under-reports.
DERIVE_EDGES_FROM_DDL = "always"

# --- execution --------------------------------------------------------------
NORMALIZE_CASING = True               # rewrite identifier casing to match target
CREATE_MISSING_SCHEMAS = False        # False => block and print the script
SKIP_DESCENDANTS_ON_FAILURE = True    # False => attempt every view regardless
SKIP_VIEWS_WITH_MISSING_BASE = True   # See note 11.
SMOKE_TEST = True                     # SELECT TOP (0) after each create
MAX_PARALLEL_PER_LEVEL = 1            # raise once you have a clean run

# The SQL analytics endpoint's metadata sync lags a CREATE. A view in level
# n+1 can fail to bind against one created moments earlier in level n.
LEVEL_SETTLE_SECONDS = 5

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_TS = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"run_id = {RUN_ID}   DRY_RUN = {DRY_RUN}")

# CELL ********************
# ## Cell 2 - read metadata, align snapshots, derive the target item set
#
# The three tables are compared for SPREAD, not equality. Writers disagree on
# precision, and a microsecond difference between tables written by the same
# extract must not orphan one of them. After filtering, a table that had rows
# and now has none blocks the run -- that is the exact signal that a filter
# literal did not match, and it is the failure this cell exists to prevent.

import pandas as pd
from datetime import timedelta
from pyspark.sql import functions as F

_v = spark.table(VIEWS_TABLE)
_d = spark.table(DEPENDENCIES_TABLE)
_o = spark.table(OBJECTS_TABLE)


def _snapshot_report(df, label):
    """Print the newest few stamps and return the latest, or None if empty."""
    rows = (df.groupBy("extracted_at_utc").count()
              .orderBy(F.col("extracted_at_utc").desc()).limit(5).collect())
    if not rows:
        print(f"  {label:14} EMPTY")
        return None
    print(f"  {label:14} " + " | ".join(
        f"{r['extracted_at_utc']} ({r['count']})" for r in rows))
    return rows[0]["extracted_at_utc"]


def _spread_seconds(stamps):
    """Seconds between earliest and latest. Blocks on mixed types.

    A string stamp compares lexically, which silently mismatches on precision
    -- '...305' and '...305652' are neither equal nor meaningfully ordered.
    Cast at the source rather than coercing here.
    """
    if not all(isinstance(s, datetime) for s in stamps):
        raise TypeError(f"extracted_at_utc is not a timestamp in every table: "
                        f"{[type(s).__name__ for s in stamps]}. Cast at the "
                        f"source; string comparison silently mismatches on "
                        f"precision.")
    return (max(stamps) - min(stamps)).total_seconds()


def _trunc_ms(ts):
    return ts.replace(microsecond=(ts.microsecond // 1000) * 1000)


def _apply_snapshot(df, stamp, label):
    """Filter to one snapshot. Blocks if the filter empties a non-empty table."""
    n_before = df.count()
    if stamp is None:
        print(f"  {label:14} {n_before} row(s), ALL snapshots")
        return df

    if SNAPSHOT_MATCH_MODE == "millisecond":
        lo = _trunc_ms(stamp)
        out = df.filter((F.col("extracted_at_utc") >= F.lit(lo))
                        & (F.col("extracted_at_utc")
                           < F.lit(lo + timedelta(milliseconds=1))))
    elif SNAPSHOT_MATCH_MODE == "exact":
        out = df.filter(F.col("extracted_at_utc") == F.lit(stamp))
    else:
        raise ValueError(f"SNAPSHOT_MATCH_MODE must be 'exact' or "
                         f"'millisecond', got {SNAPSHOT_MATCH_MODE!r}")

    n_after = out.count()
    if n_before and not n_after:
        raise ValueError(
            f"{label}: the snapshot filter {stamp!r} removed all {n_before} "
            f"row(s). The stamp does not exist in this table -- usually a "
            f"precision mismatch between writers. Set SNAPSHOT_MATCH_MODE = "
            f"'millisecond', or pin APPROVED_SNAPSHOT to a stamp the table "
            f"actually holds.")
    print(f"  {label:14} {n_after} of {n_before} row(s) @ {stamp}")
    return out


print("available snapshots (newest 5 per table):")
_sv = _snapshot_report(_v, "views")
_sd = _snapshot_report(_d, "dependencies")
_so = _snapshot_report(_o, "objects")

if APPROVED_SNAPSHOT:
    _pin = (datetime.fromisoformat(APPROVED_SNAPSHOT)
            if isinstance(APPROVED_SNAPSHOT, str) else APPROVED_SNAPSHOT)
    _sv = _sd = _so = _pin
    print(f"\npinned snapshot: {_pin}  (match mode: {SNAPSHOT_MATCH_MODE})")
elif LATEST_SNAPSHOT_ONLY:
    if None in (_sv, _sd, _so):
        raise ValueError("A metadata table is empty. Re-run the extract; do "
                         "not deploy against a partial snapshot.")
    _spread = _spread_seconds([_sv, _sd, _so])
    if _spread > SNAPSHOT_TOLERANCE_SEC:
        raise ValueError(f"Metadata tables are {_spread:.0f}s apart -- they "
                         f"are not from the same extract. See note 6.")
    if _spread:
        print(f"\nNOTE: per-table stamps differ by {_spread:.6f}s (within "
              f"tolerance). Each table uses its own latest -- a precision "
              f"difference between writers must not orphan a table.")
    else:
        print(f"\nsnapshot: {_sv}")
else:
    _sv = _sd = _so = None
    print("\nsnapshot: ALL -- every extracted_at_utc in the tables")

print("after snapshot filter:")
_v = _apply_snapshot(_v, _sv, "views")
_d = _apply_snapshot(_d, _sd, "dependencies")
_o = _apply_snapshot(_o, _so, "objects")


def _txt(value):
    """Null-safe string coercion. pd.NA raises on bool(), so no `or ''`."""
    if value is None or pd.isna(value):
        return ""
    return str(value)


def _read(df, required, optional):
    """toPandas with a required-column contract; optional columns become None.

    This is where an older extract degrades gracefully: a missing
    source_node_id, resolved_type or item_type is announced, not fatal.
    """
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Required column(s) {missing} absent. Found: "
                         f"{sorted(df.columns)}")
    pdf = df.select(*(required + [c for c in optional if c in df.columns])).toPandas()
    for c in optional:
        if c not in pdf.columns:
            print(f"NOTE: optional column '{c}' absent; degraded handling applies.")
            pdf[c] = None
    return pdf


views = _read(_v, ["node_id", "item_name", "schema_name", "view_name",
                   "view_definition"], [])
deps = _read(_d, ["source_id", "target_id"],
             ["source_node_id", "resolved_type"])
objects = _read(_o, ["item_name", "schema_name", "object_type"], ["item_type"])


def _blank(obj):
    """True where a value is null or whitespace-only.

    Accepts a Series or a DataFrame. The .str accessor exists only on Series,
    so a DataFrame is handled column by column and reassembled -- built
    explicitly rather than via .apply() so the result keeps its shape even
    when there are zero rows.
    """
    if isinstance(obj, pd.DataFrame):
        return pd.DataFrame({c: _blank(obj[c]) for c in obj.columns},
                            index=obj.index)
    return obj.isna() | (obj.astype("string").str.strip() == "")


# --- null gate on identifiers ----------------------------------------------
# A NULL identifier cannot be lowercased, cannot be matched against the target
# catalog, and cannot be safely guessed. Fail here with the offending rows
# rather than deeper in with an AttributeError. Empty strings count as null.
_key_cols = ["node_id", "item_name", "schema_name", "view_name"]
_bad = views[_blank(views[_key_cols]).any(axis=1)]
if not _bad.empty:
    print(f"BLOCKER: {len(_bad)} row(s) in {VIEWS_TABLE} have a null or empty "
          f"identifier. Re-extract the metadata; do not patch here.")
    display(_bad[_key_cols])
    raise ValueError(f"{len(_bad)} unusable row(s) in {VIEWS_TABLE}.")

_bad_deps = deps[_blank(deps[["source_id", "target_id"]]).any(axis=1)]
if not _bad_deps.empty:
    print(f"Dropping {len(_bad_deps)} dependency edge(s) with a null endpoint "
          f"-- an edge to nothing carries no ordering information.")
    deps = deps[~_blank(deps[["source_id", "target_id"]]).any(axis=1)]

_bad_obj = objects[_blank(objects[["item_name", "schema_name"]]).any(axis=1)]
if not _bad_obj.empty:
    print(f"Dropping {len(_bad_obj)} row(s) from {OBJECTS_TABLE} with a null "
          f"item_name or schema_name.")
    objects = objects[~_blank(objects[["item_name", "schema_name"]]).any(axis=1)]

# --- duplicate node_id gate -------------------------------------------------
_dupes = views["node_id"].str.lower().value_counts()
_dupes = _dupes[_dupes > 1]
if not _dupes.empty:
    print(f"BLOCKER: {len(_dupes)} node_id(s) appear more than once in the "
          f"selected snapshot. Usually two extracts in one table -- set "
          f"LATEST_SNAPSHOT_ONLY or APPROVED_SNAPSHOT.")
    display(views[views["node_id"].str.lower().isin(_dupes.index)]
            [["node_id", "item_name", "schema_name", "view_name"]])
    raise ValueError("Duplicate node_id in view_definitions.")

if INCLUDE_ITEMS:
    views = views[views["item_name"].isin(INCLUDE_ITEMS)]
if EXCLUDE_ITEMS:
    _before = len(views)
    views = views[~views["item_name"].isin(EXCLUDE_ITEMS)]
    print(f"Excluded {_before - len(views)} view(s) from {sorted(EXCLUDE_ITEMS)}")
if EXCLUDE_VIEWS:
    _drop = {e.lower() for e in EXCLUDE_VIEWS}
    _matched = set(views["node_id"].str.lower()) & _drop
    for _miss in sorted(_drop - _matched):
        print(f"NOTE: EXCLUDE_VIEWS entry {_miss!r} matched no node_id -- "
              f"check the spelling, it is not protecting anything.")
    views = views[~views["node_id"].str.lower().isin(_drop)]
    print(f"Excluded {len(_matched)} view(s) by node_id: {sorted(_matched)}")

if views.empty:
    raise ValueError("No view definitions to deploy. Check APPROVED_SNAPSHOT, "
                     "INCLUDE_ITEMS, EXCLUDE_ITEMS and EXCLUDE_VIEWS.")

# Target items come from the metadata, not from a hand-maintained list.
TARGET_ITEMS = set(views["item_name"].unique())
TARGET_ITEMS_LOWER = {t.lower(): t for t in TARGET_ITEMS}

# Lowercased keys for all graph work. See note 3 in the header.
views["key"] = views["node_id"].str.lower()
deps["source_key"] = deps["source_id"].str.lower()
deps["target_key"] = deps["target_id"].str.lower()
deps["source_node_key"] = (deps["source_node_id"].astype("string").str.strip()
                                                 .str.lower()
                                                 .replace({"": None}))

view_ids = set(views["key"])
meta = {r.key: r for r in views.itertuples(index=False)}

_have_snid = int(deps["source_node_key"].notna().sum())
print(f"\n{len(view_ids)} view(s), {len(deps)} dependency edge(s) "
      f"({_have_snid} with source_node_id, {len(deps) - _have_snid} without)")
print(f"target items derived from metadata: {sorted(TARGET_ITEMS)}")
print(views.groupby("item_name").size().to_string())

# CELL ********************
# ## Cell 3 - validate metadata, build the graph, compute levels
#
# EDGE KEY RESOLUTION, in order:
#
#   1. source_node_id, when the extract populated it. Authoritative -- it
#      carries the same #VIEW / #USER_TABLE suffix as node_id.
#   2. source_id widened to accept a single #-suffixed match. Used only where
#      the extract left source_node_id blank (OUT_OF_SCAN_SCOPE, NOT_FOUND,
#      AMBIGUOUS_CASE).
#   3. Otherwise the source is not a view in this deployment: a base object,
#      recorded for the Cell 6 pre-flight.
#   4. Then, per DERIVE_EDGES_FROM_DDL, the stored definitions themselves.
#
# Step 2 can add an ordering edge the source SQL did not intend, when a view
# and a table share a name under different casing. That is the safe direction:
# a spurious edge only delays a view, a missing one deploys it too early.

import re
from collections import defaultdict

problems = []

# A NULL definition means the extract ran without VIEW DEFINITION permission.
# Deploying it would drop the view and put nothing back.
no_ddl = [v for v in view_ids
          if meta[v].view_definition is None or pd.isna(meta[v].view_definition)]
if no_ddl:
    problems.append(f"{len(no_ddl)} view(s) have no captured DDL: {no_ddl[:5]}")

_SKIP_LIT = re.compile(r"'(?:[^']|'')*'|--[^\n]*|/\*.*?\*/", re.DOTALL)


def _mask_literals(text):
    """Blank literals and comments, preserving length so offsets stay valid."""
    buf = list(text)
    for m in _SKIP_LIT.finditer(text):
        for i in range(m.start(), m.end()):
            buf[i] = " "
    return "".join(buf)


# --- structural sanity on the captured DDL ----------------------------------
# A definition cut short by a loader fails at CREATE with a syntax error near
# whatever construct straddled the cut, which reads as a DDL bug rather than a
# capture bug. Counting outside literals makes this a warning that is never a
# false alarm.
_unbalanced = []
for v in view_ids:
    _dd = meta[v].view_definition
    if _dd is None or pd.isna(_dd):
        continue
    _mk = _mask_literals(str(_dd))
    _paren = _mk.count("(") - _mk.count(")")
    _brack = _mk.count("[") - _mk.count("]")
    _case = (len(re.findall(r"\bCASE\b", _mk, re.IGNORECASE))
             - len(re.findall(r"\bEND\b", _mk, re.IGNORECASE)))
    if _paren or _brack or _case > 0:
        _unbalanced.append((v, _paren, _brack, _case, len(str(_dd))))

if _unbalanced:
    print(f"WARNING: {len(_unbalanced)} definition(s) are structurally "
          f"unbalanced -- almost always a truncated capture. These will fail "
          f"at CREATE with a syntax error:")
    print(f"  {'node_id':50} {'()':>5} {'[]':>5} {'CASE/END':>9} {'chars':>7}")
    for v, p, b, c, n in sorted(_unbalanced):
        print(f"  {meta[v].node_id:50} {p:5} {b:5} {c:9} {n:7}")
    print("  Fix the extract or loader; rewriting the DDL here would be "
          "guessing at the missing text.")

# bare key -> the deployed view(s) sharing it, ignoring any # suffix
suffix_index = defaultdict(set)
for v in view_ids:
    suffix_index[v.split("#", 1)[0]].add(v)

parents, children = defaultdict(set), defaultdict(set)
base_refs = set()                 # bare "item.schema.object" keys, for Cell 6
how_counts = defaultdict(int)
suffix_resolved, ambiguous_edges, flagged_ambiguous = set(), [], []

for row in deps.itertuples(index=False):
    if row.target_key not in view_ids:
        continue                                   # referencing view not deployed

    snid = row.source_node_key
    if snid is not None and not pd.isna(snid):
        src = snid if snid in view_ids else None
        how = "source_node_id" if src else "base object"
    else:
        cands = suffix_index.get(row.source_key, set())
        if len(cands) == 1:
            src, how = next(iter(cands)), "suffix-resolved"
            if src != row.source_key:
                suffix_resolved.add((row.source_id, src))
        elif len(cands) > 1:
            src, how = None, "AMBIGUOUS"
            ambiguous_edges.append((row.source_id, row.target_id, sorted(cands)))
        else:
            src, how = None, "base object"

    if _txt(row.resolved_type).upper().startswith("AMBIGUOUS"):
        flagged_ambiguous.append((row.source_id, row.target_id, src))

    how_counts[how] += 1

    if src is None:
        if how == "base object":
            base_refs.add(row.source_key.split("#", 1)[0])
        continue
    if src == row.target_key:
        continue                                   # self-reference
    parents[row.target_key].add(src)
    children[src].add(row.target_key)

print("\nedge resolution: " + (", ".join(f"{k}={v}" for k, v in
                                         sorted(how_counts.items())) or "none"))

if suffix_resolved:
    print(f"\n{len(suffix_resolved)} edge(s) matched a suffixed view via "
          f"source_id (source_node_id was blank):")
    for bare, real in sorted(suffix_resolved):
        print(f"  {bare}  ->  {real}")

if flagged_ambiguous:
    print(f"\n{len(flagged_ambiguous)} edge(s) the EXTRACT marked ambiguous "
          f"(a view and a table share this name under different casing). "
          f"Resolve in the source model, not here -- under a case-sensitive "
          f"target these are two distinct objects:")
    for s, t, chosen in sorted(flagged_ambiguous):
        print(f"  {s}\n    referenced by {t}\n    ordered against: {chosen}")

if ambiguous_edges:
    problems.append(f"{len(ambiguous_edges)} edge(s) match more than one "
                    f"deployed view and were dropped: {ambiguous_edges[:3]}")

# --- edges derived from the stored DDL --------------------------------------
# The definition is the ground truth for what a view reads. Parsing it removes
# the dependence on the dependency table being both populated and correctly
# joined, and it sidesteps note 2: SQL names objects as item.schema.view,
# never with the extract's #VIEW suffix.
#
# This pattern permits spaces in unbracketed names, because this estate has
# objects such as "GL Transaction". Cell 5b uses a stricter one -- see there.

_PART_ID = r"(?:\[[^\]]+\]|[A-Za-z_][A-Za-z0-9_ ]*)"
_REF_ID = re.compile(rf"(?<![\w\].])({_PART_ID})\s*\.\s*({_PART_ID})"
                     rf"(?:\s*\.\s*({_PART_ID}))?")


def _unbracket(tok):
    return tok[1:-1] if tok.startswith("[") else tok


name_index = defaultdict(set)
for _r in views.itertuples(index=False):
    name_index[f"{_r.item_name}.{_r.schema_name}.{_r.view_name}".lower()].add(_r.key)


def ddl_refs(key):
    """(deployed-view keys this DDL reads, three-part base names it reads)."""
    m = meta[key]
    ddl = m.view_definition
    if ddl is None or pd.isna(ddl):
        return set(), set()
    vhits, bhits = set(), set()
    for mt in _REF_ID.finditer(_mask_literals(str(ddl))):
        p1, p2, p3 = (_unbracket(g).strip() if g else None for g in mt.groups())
        if p3:
            cand, qualified = f"{p1}.{p2}.{p3}".lower(), True
        else:
            # Two-part: same item. Accepted only when it matches a deployed
            # view -- an alias.column reference will not, so false positives
            # are rare and a false positive only adds an ordering edge.
            cand, qualified = f"{m.item_name}.{p1}.{p2}".lower(), False
        hit = name_index.get(cand)
        if hit:
            vhits |= hit
        elif qualified and p1.lower() in TARGET_ITEMS_LOWER:
            bhits.add(cand)
    return vhits - {key}, bhits


if DERIVE_EDGES_FROM_DDL not in ("off", "fallback", "always"):
    raise ValueError(f"DERIVE_EDGES_FROM_DDL must be 'off', 'fallback' or "
                     f"'always', got {DERIVE_EDGES_FROM_DDL!r}")

if (DERIVE_EDGES_FROM_DDL == "always"
        or (DERIVE_EDGES_FROM_DDL == "fallback" and not any(parents.values()))):
    _added, _base_before = 0, len(base_refs)
    for _tgt in view_ids:
        _vh, _bh = ddl_refs(_tgt)
        for _src in _vh:
            if _src not in parents[_tgt]:
                _added += 1
            parents[_tgt].add(_src)
            children[_src].add(_tgt)
        base_refs |= _bh
    print(f"\nDDL parse: +{_added} view-to-view edge(s) beyond the dependency "
          f"table, +{len(base_refs) - _base_before} base object(s)")

if REQUIRE_DEPENDENCY_EDGES and len(view_ids) > 1 and not any(parents.values()):
    problems.append(
        "No view-to-view edges resolved. Every view would deploy in one wave, "
        "and any view reading another would fail at CREATE. Check the "
        "dependencies table snapshot -- Cell 2 prints per-table stamps.")


def dependency_levels(nodes):
    """Group nodes into waves. Level 0 depends only on non-deployed objects.

    Returns (levels, unplaced). A non-empty unplaced set means a cycle.
    """
    nodes = set(nodes)
    remaining = {n: len(parents[n] & nodes) for n in nodes}
    placed, levels = set(), []
    while True:
        wave = sorted(n for n, d in remaining.items()
                      if n not in placed and d == 0)
        if not wave:
            break
        levels.append(wave)
        placed.update(wave)
        for n in wave:
            for c in children[n] & nodes:
                remaining[c] -= 1
    return levels, nodes - placed


levels, cyclic = dependency_levels(view_ids)
if cyclic:
    # Views cannot genuinely be circular, so this means stale metadata --
    # usually an object dropped and recreated after extraction. It can also be
    # the DDL parser matching a self-reference through an alias; if the cycle
    # only appears under 'always', drop back to 'fallback'.
    problems.append(f"Dependency cycle among {len(cyclic)} view(s): "
                    f"{sorted(cyclic)[:10]}. Re-extract the metadata, or try "
                    f"DERIVE_EDGES_FROM_DDL = 'fallback'.")

if problems:
    for p in problems:
        print(f"BLOCKER: {p}")
    raise ValueError(f"{len(problems)} validation failure(s). Nothing deployed.")

print(f"\n{sum(len(v) for v in parents.values())} view-to-view edge(s), "
      f"{len(base_refs)} distinct object(s) referenced but not deployed")
print(f"{len(levels)} dependency level(s):")
for i, wave in enumerate(levels):
    print(f"  level {i}: {len(wave)} view(s)")
    if DRY_RUN:
        for n in wave:
            print(f"      {meta[n].node_id}")

# CELL ********************
# ## Cell 4 - build the deployment plan

plan = []
for level_no, wave in enumerate(levels):
    for key in wave:
        m = meta[key]
        plan.append({
            "key": key,                  # lowercased, for graph lookups
            "node_id": m.node_id,        # original casing, for the log
            "level": level_no,
            "target_item": m.item_name,
            "schema_name": m.schema_name,
            "view_name": m.view_name,
            "ddl": m.view_definition,
            "ddl_stored": m.view_definition,   # never mutated
        })

plan_pdf = pd.DataFrame(plan)
print(f"{len(plan)} view(s) planned across {len(levels)} level(s)")
if DRY_RUN:
    display(plan_pdf[["level", "target_item", "schema_name", "view_name"]])

# CELL ********************
# ## Cell 5 - connections (thread-local) and target existence check

import struct
import threading

import pyodbc
import sempy.fabric as fabric
from notebookutils import credentials

_ws_id = (fabric.resolve_workspace_id(TARGET_WORKSPACE) if TARGET_WORKSPACE
          else fabric.get_workspace_id())
_tls = threading.local()
_endpoints = None


def _endpoint_map():
    """lowercased display name -> (real display name, connection string)."""
    global _endpoints
    if _endpoints is not None:
        return _endpoints
    client = fabric.FabricRestClient()
    out = {}
    for route in ("lakehouses", "warehouses"):
        for it in client.get(f"/v1/workspaces/{_ws_id}/{route}").json()["value"]:
            p = it.get("properties", {})
            cs = (p.get("connectionString")
                  or p.get("sqlEndpointProperties", {}).get("connectionString"))
            if cs:
                out[it["displayName"].lower()] = (it["displayName"], cs)
    _endpoints = out
    return out


def connect(item_name):
    """This thread's connection to item_name, opened on first use.

    One connection per thread per item: Fabric has no MARS, so a shared
    connection cannot carry concurrent statements. See note 1 in the header.
    """
    if not hasattr(_tls, "conns"):
        _tls.conns = {}
    lookup = item_name.lower()
    if lookup in _tls.conns:
        return _tls.conns[lookup]

    entry = _endpoint_map().get(lookup)
    if entry is None:
        raise RuntimeError(f"'{item_name}' has no SQL endpoint in the target "
                           f"workspace.")
    real_name, server = entry
    tok = credentials.getToken(
        "https://analysis.windows.net/powerbi/api").encode("UTF-16-LE")
    tok_struct = struct.pack(f"<I{len(tok)}s", len(tok), tok)
    cstr = (f"Driver={{ODBC Driver 18 for SQL Server}};Server={server},1433;"
            f"Database={real_name};Encrypt=yes;TrustServerCertificate=no;")
    conn = pyodbc.connect(cstr, attrs_before={1256: tok_struct},
                          autocommit=True)
    _tls.conns[lookup] = conn
    return conn


# Every item named in the metadata must exist in the target workspace. This
# replaces a hand-maintained allowlist: an item that is not here is either a
# source-only sandbox (exclude it) or genuinely not provisioned yet.
absent_items = sorted(i for i in TARGET_ITEMS
                      if i.lower() not in _endpoint_map())
if absent_items:
    print("BLOCKER: these items appear in the metadata but have no SQL "
          "endpoint in the target workspace:")
    for i in absent_items:
        print(f"  {i}")
    print("\nEither provision them, or add them to EXCLUDE_ITEMS in Cell 1.")
    print(f"\nAvailable in the target workspace: "
          f"{sorted(v[0] for v in _endpoint_map().values())}")
    raise ValueError(f"{len(absent_items)} target item(s) missing. "
                     f"Nothing deployed.")

for item in sorted(TARGET_ITEMS):
    cur = connect(item).cursor()
    cur.execute("SELECT collation_name FROM sys.databases WHERE name = DB_NAME()")
    collation = (cur.fetchone() or [None])[0] or "unknown"
    sensitivity = "CASE-SENSITIVE" if "_BIN2" in collation else "case-insensitive"
    print(f"  {item:24} {collation}  ({sensitivity})")

print("\nCollation cannot be changed after an item is created. If the source "
      "environment is case-insensitive and the target is not, identifier "
      "casing in the stored DDL needs the normalization in Cell 5b.")

# CELL ********************
# ## Cell 5a - schema gate (creates nothing by default)
#
# CREATE VIEW fails with error 2760 if the schema does not exist; schemas are
# never created implicitly. This blocks and prints the script to run yourself,
# keeping one-time setup separate from repeatable deployment. Once the schemas
# exist the cell is silent.
#
# Fabric has two kinds of schema, and they are not interchangeable:
#
#   Lakehouse schemas     created on the Spark/lakehouse side, hold Delta
#                         tables, surface in the SQL endpoint automatically.
#                         Must come from your data pipelines -- a same-named
#                         SQL-only schema does not exist in OneLake and can
#                         collide with the sync.
#
#   SQL endpoint schemas  created with T-SQL, hold views/procedures/functions
#                         only, invisible in Lakehouse Explorer and OneLake.
#                         The correct home for deployed views.
#
# A WAREHOUSE has only the second kind -- hence the item_type check. See
# note 5.
#
# CASING NOTE: the comparison is case-insensitive on purpose. A schema that
# exists as "contoso" satisfies a plan entry spelled "Contoso"; Cell 5b
# corrects the spelling. Creating a second schema differing only in case is
# never the right repair.

_needed = {(p["target_item"], p["schema_name"]) for p in plan
           if p["schema_name"].lower() != "dbo"}

# A schema holding at least one table in the source is a lakehouse schema.
_table_schemas = {(r.item_name.lower(), r.schema_name.lower())
                  for r in objects.itertuples(index=False)
                  if r.object_type == "USER_TABLE"}

_warehouse_items = {r.item_name.lower() for r in objects.itertuples(index=False)
                    if _txt(r.item_type).upper() == "WAREHOUSE"}
if not _warehouse_items:
    print("NOTE: no item_type = WAREHOUSE found. If a target IS a warehouse, "
          "its schemas will be misclassified as pipeline-owned.")

missing_view_only, missing_table_bearing = defaultdict(list), defaultdict(list)
for item in sorted({i for i, _ in _needed}):
    cur = connect(item).cursor()
    cur.execute("SELECT name FROM sys.schemas")
    present = {r[0].lower() for r in cur.fetchall()}
    for _i, schema in sorted(s for s in _needed if s[0] == item):
        if schema.lower() in present:
            continue
        if ((item.lower(), schema.lower()) in _table_schemas
                and item.lower() not in _warehouse_items):
            missing_table_bearing[item].append(schema)
        else:
            missing_view_only[item].append(schema)

if missing_table_bearing:
    n = sum(len(v) for v in missing_table_bearing.values())
    print(f"{n} missing schema(s) hold TABLES in the source. Do not create "
          f"these by hand -- run the pipelines that populate them:")
    for item, schemas in sorted(missing_table_bearing.items()):
        for s in schemas:
            print(f"  {item}.{s}")

if missing_view_only:
    n = sum(len(v) for v in missing_view_only.values())
    print(f"\n{n} missing view-only schema(s). Run this once per item, in "
          f"that item's SQL endpoint. Keep the GO separators: CREATE SCHEMA "
          f"must be alone in its batch.\n")
    for item, schemas in sorted(missing_view_only.items()):
        print(f"-- ---------- {item} ----------")
        for s in schemas:
            print(f"CREATE SCHEMA [{s}];\nGO")
        print()
    if CREATE_MISSING_SCHEMAS and not DRY_RUN:
        for item, schemas in sorted(missing_view_only.items()):
            for s in schemas:
                connect(item).cursor().execute(f"CREATE SCHEMA [{s}]")
                print(f"  created {item}.{s}")
        missing_view_only.clear()

if missing_view_only or missing_table_bearing:
    raise ValueError("Missing schemas. Create them, then re-run. "
                     "Nothing deployed.")
print("All target schemas exist.")

# CELL ********************
# ## Cell 5b - normalize identifier casing against the live target catalog
#
# Every rewrite is validated against sys.objects / sys.schemas in the target:
# a name changes only when a case-insensitive match exists and the exact-case
# form does not. Nothing is guessed.
#
# TWO PASSES, TWO TERRITORIES (note 10). fix_create owns the CREATE clause;
# fix_ref owns the body, and they run on disjoint slices of the text. Running
# fix_ref over the whole definition lets it re-resolve the CREATE target
# through the seeded catalog and revert what fix_create just fixed -- error
# 2760 against a schema that does not exist.
#
# THE SEED IS BOUND TO THE LIVE SCHEMA SPELLING (note 9).
#
# IDENTIFIER PATTERN. _PART here does NOT permit spaces in unbracketed names,
# unlike the parser in Cell 3. This pattern has to tell a schema-qualified
# object from an alias.column, and loosening it makes "SELECT a.Foo Bar" parse
# as an object reference. Names containing spaces must be bracketed in the
# stored DDL to be rewritten -- unbracketed they would not be valid T-SQL.
#
# COLUMNS ARE NOT REWRITTEN. See note 12.

_PART = r"(?:\[[^\]]+\]|[A-Za-z_][A-Za-z0-9_]*)"
_REF = re.compile(rf"(?<![\w\].])({_PART})\s*\.\s*({_PART})"
                  rf"(?:\s*\.\s*({_PART}))?")
_CREATE = re.compile(rf"(CREATE\s+(?:OR\s+ALTER\s+)?VIEW\s+)({_PART})"
                     rf"(\s*\.\s*)({_PART})", re.IGNORECASE)

_cat_lock = threading.RLock()
_cat_obj, _cat_sch = {}, {}


def _load_catalog(item):
    with _cat_lock:
        if item in _cat_obj:
            return
        cur = connect(item).cursor()
        cur.execute("SELECT name FROM sys.schemas")
        schemas = {r[0].lower(): r[0] for r in cur.fetchall()}
        cur.execute("SELECT s.name, o.name FROM sys.objects AS o "
                    "JOIN sys.schemas AS s ON s.schema_id = o.schema_id "
                    "WHERE o.type IN ('U', 'V')")
        objs = {f"{s}.{n}".lower(): (s, n) for s, n in cur.fetchall()}
        _cat_sch[item] = schemas
        _cat_obj[item] = objs


def _bare(tok):
    return tok[1:-1] if tok.startswith("[") else tok


def _fmt(name, original):
    """Re-emit a name, bracketing if it was bracketed or if it needs it."""
    if original.startswith("[") or not re.fullmatch(r"[A-Za-z_]\w*", name):
        return f"[{name}]"
    return name


def normalize_ddl(ddl, current_item):
    """Return (new_ddl, changes, absent). Catalog-confirmed rewrites only."""
    changes, absent = [], []

    def fix_create(m):
        _load_catalog(current_item)
        want = _bare(m.group(2))
        real = _cat_sch[current_item].get(want.lower())
        if real is None or real == want:
            return m.group(0)
        changes.append((f"CREATE VIEW {want}.", f"CREATE VIEW {real}."))
        return m.group(1) + _fmt(real, m.group(2)) + m.group(3) + m.group(4)

    def fix_ref(m):
        p1, p2, p3 = m.group(1), m.group(2), m.group(3)
        if p3:                                    # item.schema.object
            real_item = TARGET_ITEMS_LOWER.get(_bare(p1).lower())
            if real_item is None:
                return m.group(0)
            _load_catalog(real_item)
            hit = _cat_obj[real_item].get(f"{_bare(p2)}.{_bare(p3)}".lower())
            if hit is None:
                absent.append(f"{_bare(p1)}.{_bare(p2)}.{_bare(p3)}")
                return m.group(0)
            if (_bare(p1), _bare(p2), _bare(p3)) == (real_item, *hit):
                return m.group(0)
            new = (f"{_fmt(real_item, p1)}.{_fmt(hit[0], p2)}"
                   f".{_fmt(hit[1], p3)}")
            changes.append((m.group(0).strip(), new))
            return new

        # Two-part. Guarded: part one must be a real SCHEMA name, which an
        # alias such as "a" or "src" will not be. Without this guard,
        # "a.ColumnName" would be treated as a schema-qualified object.
        _load_catalog(current_item)
        if _bare(p1).lower() not in _cat_sch[current_item]:
            return m.group(0)
        hit = _cat_obj[current_item].get(f"{_bare(p1)}.{_bare(p2)}".lower())
        if hit is None or (_bare(p1), _bare(p2)) == hit:
            return m.group(0)
        new = f"{_fmt(hit[0], p1)}.{_fmt(hit[1], p2)}"
        changes.append((m.group(0).strip(), new))
        return new

    m_create = _CREATE.search(ddl)
    if m_create:
        head = _CREATE.sub(fix_create, ddl[:m_create.end()], count=1)
        ddl = head + _REF.sub(fix_ref, ddl[m_create.end():])
    else:
        print(f"NOTE: no CREATE VIEW clause found in a definition for "
              f"{current_item}; the reference pass ran over the whole text.")
        ddl = _REF.sub(fix_ref, ddl)
    return ddl, changes, absent


def _seed_planned_views():
    """Register views this run creates, bound to the live schema spelling.

    setdefault, never overwrite: an object that already exists in the target
    is authoritative over what this run intends to create.
    """
    seeded = 0
    with _cat_lock:
        for step in plan:
            item = step["target_item"]
            _load_catalog(item)
            real_sch = _cat_sch[item].get(step["schema_name"].lower(),
                                          step["schema_name"])
            okey = f"{real_sch}.{step['view_name']}".lower()
            if okey not in _cat_obj[item]:
                _cat_obj[item][okey] = (real_sch, step["view_name"])
                seeded += 1
            _cat_sch[item].setdefault(step["schema_name"].lower(),
                                      step["schema_name"])
    return seeded


# Keys whose sources Cell 5b proved missing. Cell 7 skips these rather than
# dropping a working view to replace it with a guaranteed failure.
blocked_by_missing = set()

if NORMALIZE_CASING:
    print(f"Seeded {_seed_planned_views()} not-yet-created view(s) into the "
          f"casing catalog, bound to the live schema spelling.")

    all_changes, all_absent, touched = [], {}, 0
    for step in plan:
        new_ddl, changes, absent = normalize_ddl(step["ddl"],
                                                 step["target_item"])
        if changes:
            step["ddl"] = new_ddl
            touched += 1
            all_changes.extend(changes)
        # Keep DROP VIEW in step with the rewritten CREATE.
        real_sch = _cat_sch.get(step["target_item"], {}).get(
            step["schema_name"].lower())
        if real_sch and real_sch != step["schema_name"]:
            step["schema_name"] = real_sch
        for a in absent:
            all_absent.setdefault(a, []).append(step["key"])

    print(f"Casing normalized in {touched} of {len(plan)} view(s), "
          f"{len(all_changes)} identifier(s) rewritten")
    for old, new in sorted(set(all_changes)):
        print(f"  {old}  ->  {new}")

    if all_absent:
        pending = {a for a in all_absent
                   if a.lower() in view_ids or a.lower() in suffix_index}
        real = {a: u for a, u in all_absent.items() if a not in pending}
        if pending:
            print(f"\n{len(pending)} reference(s) point at views this run "
                  f"creates later -- expected, the levels handle ordering.")
        if real:
            print(f"\n{len(real)} reference(s) exist in NEITHER the target nor "
                  f"this deployment. Views using them cannot succeed:")
            for ref, users in sorted(real.items(), key=lambda kv: -len(kv[1])):
                print(f"  {ref:55} blocks {len(users)} view(s)")
                blocked_by_missing.update(users)
            print("\nEach is either an object your pipelines have not created "
                  "in the target, or a view missing from the metadata "
                  "snapshot -- in which case re-extract before deploying.")
            if SKIP_VIEWS_WITH_MISSING_BASE:
                print(f"\n{len(blocked_by_missing)} view(s) will be SKIPPED "
                      f"rather than dropped and failed (note 11):")
                for k in sorted(blocked_by_missing):
                    print(f"  {meta[k].node_id}")
        else:
            print("\nEvery unresolved reference is satisfied within this run.")
else:
    print("NORMALIZE_CASING is off; DDL executes exactly as stored.")

# CELL ********************
# ## Cell 6 - pre-flight: do the referenced base objects exist?
#
# Views over objects that have not landed yet fail at CREATE. Checking first
# turns a missing pipeline run into one clear list instead of N errors.
#
# base_refs comes from Cell 3, so this uses the SAME edge resolution as the
# level computation -- including DDL-derived references. Deriving it again
# from source_id here would re-introduce the suffix mismatch and report
# deployed views as missing base objects.
#
# Both tables AND views count as present: a base object may legitimately be a
# view that exists in the target but is not part of this deployment. Checking
# sys.tables alone reports every such view as missing.
#
# A NOTE ON COLUMN ERRORS. This pre-flight checks objects, not columns. If a
# view fails with error 207 (invalid column name) after every object resolves,
# the definition disagrees with the columns the source table actually exposes
# -- a name difference, not a casing one, and usually created at ingestion.
# The repair is in the loader or the source view, not here.

base_needed = defaultdict(set)
for ref in base_refs:
    parts = ref.split(".", 2)
    if len(parts) == 3:
        base_needed[parts[0]].add((parts[1], parts[2]))

# Views this run creates are not base objects, whatever the metadata called
# them -- they are satisfied by the level ordering.
_planned_names = {(p["target_item"].lower(), p["schema_name"].lower(),
                   p["view_name"].lower()) for p in plan}

missing_base = []
for item_l, wanted in sorted(base_needed.items()):
    real_item = TARGET_ITEMS_LOWER.get(item_l)
    if real_item is None:
        # Not a deploy target: could be a staging item or another workspace.
        # Nothing to check against, so report rather than assume.
        missing_base.extend(f"{item_l}.{s}.{o} (item not a deploy target)"
                            for s, o in sorted(wanted))
        continue
    cur = connect(real_item).cursor()
    cur.execute("SELECT s.name, o.name FROM sys.objects AS o "
                "JOIN sys.schemas AS s ON s.schema_id = o.schema_id "
                "WHERE o.type IN ('U', 'V')")
    present = {(r[0].lower(), r[1].lower()) for r in cur.fetchall()}
    for schema, obj in sorted(wanted):
        if (schema, obj) in present:
            continue
        if (item_l, schema, obj) in _planned_names:
            continue                        # created by this run
        missing_base.append(f"{real_item}.{schema}.{obj}")

if missing_base:
    print(f"WARNING: {len(missing_base)} referenced base object(s) are not "
          f"present in the target. Views using them cannot succeed:")
    for m in missing_base[:30]:
        print(f"  {m}")
    if len(missing_base) > 30:
        print(f"  ... and {len(missing_base) - 30} more")
    print("\nNote: under a BIN2 collation a casing difference reads as absent. "
          "Check the Cell 5b rewrite list before assuming a pipeline gap.")
else:
    print(f"All {len(base_refs)} referenced base object(s) present in the target.")

# CELL ********************
# ## Cell 7 - deploy, level by level

import time
from concurrent.futures import ThreadPoolExecutor

results = {}          # key -> (status, error)


def descendants(node):
    seen, stack = set(), [node]
    while stack:
        n = stack.pop()
        for c in children[n] - seen:
            seen.add(c)
            stack.append(c)
    return seen


def _try_ddl(step, ddl):
    """Drop then create, as two batches: CREATE VIEW must be first in its own."""
    cur = connect(step["target_item"]).cursor()
    cur.execute(
        f"DROP VIEW IF EXISTS [{step['schema_name']}].[{step['view_name']}]")
    cur.execute(ddl)
    if SMOKE_TEST:
        cur.execute(f"SELECT TOP (0) * FROM "
                    f"[{step['schema_name']}].[{step['view_name']}]")
        cur.fetchall()


def deploy_one(step):
    """Deploy one view, falling back to the stored DDL if the rewrite fails.

    Casing normalization is a heuristic. If a rewrite is rejected, retrying
    the stored definition guarantees rewriting can never do worse than leaving
    it alone.

    If both attempts fail the view is left dropped -- see the header warning.
    """
    label = f"{step['target_item']}.{step['schema_name']}.{step['view_name']}"
    try:
        _try_ddl(step, step["ddl"])
        return step["key"], "OK", None, label
    except Exception as exc:                       # noqa: BLE001
        first = f"{type(exc).__name__}: {exc}"

    original = step.get("ddl_stored")
    if original is None or original == step["ddl"]:
        return step["key"], "FAILED", first, label

    try:
        _try_ddl(step, original)
        return (step["key"], "OK",
                f"rewrite rejected, stored DDL used ({first})", label)
    except Exception as exc:                       # noqa: BLE001
        return (step["key"], "FAILED",
                f"rewritten: {first} || stored: {type(exc).__name__}: {exc}",
                label)


if SKIP_VIEWS_WITH_MISSING_BASE and blocked_by_missing:
    for _k in blocked_by_missing:
        results[_k] = ("SKIPPED", "source object exists in neither the target "
                                  "nor this run")
    print(f"{len(blocked_by_missing)} view(s) pre-marked SKIPPED "
          f"(missing source objects); they will not be dropped.")

if DRY_RUN:
    print("DRY_RUN: nothing deployed. Review the output above, then set "
          "DRY_RUN = False.")
else:
    for level_no, wave in enumerate(levels):
        steps = [p for p in plan
                 if p["level"] == level_no and p["key"] not in results]
        if not steps:
            continue

        # The endpoint's metadata sync lags the CREATE. Without a settle, a
        # level n+1 view can fail to bind against a level n view that
        # demonstrably exists -- which reads as a graph bug and is not.
        if level_no and LEVEL_SETTLE_SECONDS:
            print(f"  settling {LEVEL_SETTLE_SECONDS}s for endpoint metadata "
                  f"sync before level {level_no}")
            time.sleep(LEVEL_SETTLE_SECONDS)

        print(f"\nLevel {level_no}: deploying {len(steps)} view(s)")
        with ThreadPoolExecutor(max_workers=MAX_PARALLEL_PER_LEVEL) as pool:
            for key, status, err, label in pool.map(deploy_one, steps):
                results[key] = (status, err)
                print(f"  {status:7} {label}" + (f"  -- {err}" if err else ""))

        # A failed view leaves its subtree unbuildable. Marking descendants
        # SKIPPED keeps one root cause in the log instead of cascading noise.
        if SKIP_DESCENDANTS_ON_FAILURE:
            for key, (status, _) in list(results.items()):
                if status != "FAILED":
                    continue
                for d in descendants(key) & view_ids:
                    if d not in results:
                        results[d] = ("SKIPPED", f"upstream failed: {key}")

# CELL ********************
# ## Cell 8 - run log and summary

from pyspark.sql.types import (IntegerType, StringType, StructField,
                               StructType, TimestampType)

LOG_SCHEMA = StructType([
    StructField("run_id", StringType()),
    StructField("run_ts_utc", TimestampType()),
    StructField("node_id", StringType()),
    StructField("target_item", StringType()),
    StructField("schema_name", StringType()),
    StructField("view_name", StringType()),
    StructField("level", IntegerType()),
    StructField("status", StringType()),
    StructField("error", StringType()),
])

rows = []
for step in plan:
    status, err = results.get(step["key"], ("NOT_ATTEMPTED", None))
    rows.append({
        "run_id": RUN_ID, "run_ts_utc": RUN_TS, "node_id": step["node_id"],
        "target_item": step["target_item"], "schema_name": step["schema_name"],
        "view_name": step["view_name"], "level": int(step["level"]),
        "status": status, "error": err,
    })

log_pdf = pd.DataFrame(rows)
counts = log_pdf["status"].value_counts().to_dict()
print(f"\nrun_id {RUN_ID}: {counts}")

if not DRY_RUN:
    log_pdf["error"] = log_pdf["error"].where(log_pdf["error"].notna(), None)
    (spark.createDataFrame(log_pdf[[f.name for f in LOG_SCHEMA.fields]],
                           schema=LOG_SCHEMA)
          .write.format("delta").mode("append").saveAsTable(LOG_TABLE))
    print(f"Appended {len(log_pdf)} row(s) to {LOG_TABLE}")

failed = log_pdf[log_pdf["status"] == "FAILED"]
if not failed.empty:
    print(f"\n{len(failed)} failure(s) -- root causes only, descendants were "
          f"skipped:")
    for r in failed.itertuples(index=False):
        print(f"  {r.node_id}: {r.error}")
    print("\nError 208 (invalid object name) after Cell 5b ran means the "
          "object is genuinely absent. Error 207 (invalid column name) means "
          "the definition disagrees with the source table's columns -- fix "
          "that at ingestion, not here.")
    raise RuntimeError(f"Deployment incomplete: {counts}. Fix root causes and "
                       f"re-run; the process is idempotent.")